In [8]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 30)

# The notebook lives in notebooks/, so the project root is one level up
def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        if (folder / "data" / "raw").is_dir():
            return folder
    raise FileNotFoundError("Could not find a folder containing data/raw/ above " + str(start))

PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"

print("Project root:", PROJECT_ROOT)
print("Files found:", sorted(p.name for p in RAW_DIR.glob("*.txt")))

Project root: /home/dark/Projects/turbofan-predictive-maintenance
Files found: ['RUL_FD001.txt', 'test_FD001.txt', 'train_FD001.txt']


In [9]:
index_names = ["unit", "cycle"]
setting_names = [f"setting_{i}" for i in range(1, 4)]
sensor_names = [f"sensor_{i}" for i in range(1, 22)]
col_names = index_names + setting_names + sensor_names

print(len(col_names), "columns")

train = pd.read_csv(RAW_DIR / "train_FD001.txt", sep=r"\s+", header=None, names=col_names)
test = pd.read_csv(RAW_DIR / "test_FD001.txt", sep=r"\s+", header=None, names=col_names)
rul_test = pd.read_csv(RAW_DIR / "RUL_FD001.txt", sep=r"\s+", header=None, names=["RUL"])

print("train:", train.shape)
print("test:", test.shape)
print("rul_test:", rul_test.shape)

26 columns
train: (20631, 26)
test: (13096, 26)
rul_test: (100, 1)


In [10]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 20631 entries, 0 to 20630
Data columns (total 26 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   unit       20631 non-null  int64  
 1   cycle      20631 non-null  int64  
 2   setting_1  20631 non-null  float64
 3   setting_2  20631 non-null  float64
 4   setting_3  20631 non-null  float64
 5   sensor_1   20631 non-null  float64
 6   sensor_2   20631 non-null  float64
 7   sensor_3   20631 non-null  float64
 8   sensor_4   20631 non-null  float64
 9   sensor_5   20631 non-null  float64
 10  sensor_6   20631 non-null  float64
 11  sensor_7   20631 non-null  float64
 12  sensor_8   20631 non-null  float64
 13  sensor_9   20631 non-null  float64
 14  sensor_10  20631 non-null  float64
 15  sensor_11  20631 non-null  float64
 16  sensor_12  20631 non-null  float64
 17  sensor_13  20631 non-null  float64
 18  sensor_14  20631 non-null  float64
 19  sensor_15  20631 non-null  float64
 20  sensor_16  20631 

In [11]:
train.describe().T

,count,mean,std,min,25%,50%,75%,max
unit,20631.0,51.506568,2.922763e+01,1.0000,26.0000,52.0000,77.0000,100.0000
cycle,20631.0,108.807862,6.888099e+01,1.0000,52.0000,104.0000,156.0000,362.0000
setting_1,20631.0,-0.000009,2.187313e-03,-0.0087,-0.0015,0.0000,0.0015,0.0087
setting_2,20631.0,0.000002,2.930621e-04,-0.0006,-0.0002,0.0000,0.0003,0.0006
setting_3,20631.0,100.000000,0.000000e+00,100.0000,100.0000,100.0000,100.0000,100.0000
sensor_1,20631.0,518.670000,0.000000e+00,518.6700,518.6700,518.6700,518.6700,518.6700
sensor_2,20631.0,642.680934,5.000533e-01,641.2100,642.3250,642.6400,643.0000,644.5300
sensor_3,20631.0,1590.523119,6.131150e+00,1571.0400,1586.2600,1590.1000,1594.3800,1616.9100
sensor_4,20631.0,1408.933782,9.000605e+00,1382.2500,1402.3600,1408.0400,1414.5550,1441.4900
sensor_5,20631.0,14.620000,5.329200e-15,14.6200,14.6200,14.6200,14.6200,14.6200


In [12]:
print("Number of engines:", train["unit"].nunique())

life = train.groupby("unit")["cycle"].max()
print(life.describe())
print("Shortest life:", life.idxmin(), "->", life.min(), "cycles")
print("Longest life: ", life.idxmax(), "->", life.max(), "cycles")

Number of engines: 100
count    100.000000
mean     206.310000
std       46.342749
min      128.000000
25%      177.000000
50%      199.000000
75%      229.250000
max      362.000000
Name: cycle, dtype: float64
Shortest life: 39 -> 128 cycles
Longest life:  69 -> 362 cycles


In [13]:
test_last_cycle = test.groupby("unit")["cycle"].max()
print(test_last_cycle.describe())
print()
print(rul_test["RUL"].describe())

count    100.000000
mean     130.960000
std       53.593479
min       31.000000
25%       88.750000
50%      133.500000
75%      164.250000
max      303.000000
Name: cycle, dtype: float64

count    100.00000
mean      75.52000
std       41.76497
min        7.00000
25%       32.75000
50%       86.00000
75%      112.25000
max      145.00000
Name: RUL, dtype: float64


In [14]:
check = train[setting_names + sensor_names].agg(["nunique", "min", "max", "std"]).T
check.sort_values("nunique").head(10)

,nunique,min,max,std
sensor_1,1.0,518.6700,518.6700,0.000000e+00
setting_3,1.0,100.0000,100.0000,0.000000e+00
sensor_5,1.0,14.6200,14.6200,5.329200e-15
sensor_10,1.0,1.3000,1.3000,0.000000e+00
sensor_18,1.0,2388.0000,2388.0000,0.000000e+00
sensor_19,1.0,100.0000,100.0000,0.000000e+00
sensor_16,1.0,0.0300,0.0300,3.469531e-18
sensor_6,2.0,21.6000,21.6100,1.388985e-03
sensor_17,13.0,388.0000,400.0000,1.548763e+00
setting_2,13.0,-0.0006,0.0006,2.930621e-04
